# [12-4강] 필터 수/커널 크기 변경 실험 - 실습

In [1]:
import torch
torch.set_num_threads(1)
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import random

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

torch.set_printoptions(precision=4, sci_mode=False)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)


device: cpu


## 문제 1. CNN variant 생성 함수 만들기

filter 수와 kernel size를 입력받아 CNN 모델을 만드는 함수를 작성합니다.

In [2]:
def make_cnn(filters=4, kernel_size=3):
    # TODO: 다음 세 값을 함수 인자에 맞게 수정하세요.
    used_filters = filters
    used_kernel = kernel_size
    padding = kernel_size // 2
    return nn.Sequential(
        nn.Conv2d(1, used_filters, kernel_size=used_kernel, padding=padding),
        nn.ReLU(), nn.Flatten(),
        nn.Linear(used_filters * 8 * 8, 2)
    )

def count_params(model):
    return sum(p.numel() for p in model.parameters())
for cfg in [(4, 3), (8, 3), (8, 5)]:
    try:
        m = make_cnn(*cfg)
        print(cfg, count_params(m))
    except Exception as e:
        print(cfg, '수정 필요:', e)


(4, 3) 554
(8, 3) 1106
(8, 5) 1234


### 해설 및 실행 결과 해석

- filter 수가 늘면 Conv 출력 channel과 classifier 입력 차원이 함께 커져 parameter 수가 증가합니다. kernel size가 커져도 Conv layer parameter가 증가합니다.

## 문제 2. variant별 한 step loss 기록하기

같은 batch로 여러 CNN variant의 한 step loss를 비교합니다.

In [4]:
def make_toy_images(n=48, size=8):
    # class 0: 세로선, class 1: 가로선
    x = torch.zeros(n, 1, size, size)
    y = torch.zeros(n, dtype=torch.long)
    for i in range(n):
        if i % 2 == 0:
            x[i, 0, :, 3:5] = 1.0
            y[i] = 0
        else:
            x[i, 0, 3:5, :] = 1.0
            y[i] = 1
    x += 0.05 * torch.randn_like(x)
    return x, y

images, labels = make_toy_images()
train_ds = TensorDataset(images[:40], labels[:40])
valid_ds = TensorDataset(images[40:], labels[40:])
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
valid_loader = DataLoader(valid_ds, batch_size=8, shuffle=False)

def make_cnn(filters=4, kernel_size=3):
    if kernel_size <= 0 or kernel_size % 2 == 0:
        raise ValueError('공간 크기를 유지하려면 양의 홀수 kernel_size를 사용하세요.')
    padding = kernel_size // 2
    return nn.Sequential(
        nn.Conv2d(1, filters, kernel_size, padding=padding),
        nn.ReLU(),
        nn.Flatten(),
        nn.Linear(filters * 8 * 8, 2),
    ).to(device)

def count_params(model):
    return sum(p.numel() for p in model.parameters())

xb, yb = next(iter(train_loader)); xb, yb = xb.to(device), yb.to(device)
rows = []
for filters, kernel in [(4, 3), (8, 3), (8, 5)]:
    model = make_cnn(filters, kernel)
    # TODO: loss_fn과 optimizer를 만들고 한 step 학습하세요.
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    optimizer.zero_grad()
    logits = model(xb)
    loss = loss_fn(logits, yb)
    loss.backward()
    optimizer.step()
    rows.append({'filters': filters, 'kernel': kernel, 'params': count_params(model), 'one_step_loss': round(loss_value, 4)})
print(rows)


[{'filters': 4, 'kernel': 3, 'params': 554, 'one_step_loss': 0.0}, {'filters': 8, 'kernel': 3, 'params': 1106, 'one_step_loss': 0.0}, {'filters': 8, 'kernel': 5, 'params': 1234, 'one_step_loss': 0.0}]


### 해설 및 실행 결과 해석

- one-step loss는 실행 경로를 확인하는 관찰값입니다. 초기화의 영향이 커서 모델 선택 기준으로 쓰지 않으며, 실제 비교에서는 같은 학습 조건의 validation metric을 사용합니다.


## 문제 3. 실험 결과에서 가장 작은 one-step loss 기록 찾기

여러 variant의 결과 리스트에서 현재 기록된 one-step loss가 가장 작은 설정을 코드로 찾습니다. 이 값은 코드 동작을 확인하는 관찰값이며, best 모델 선택에는 validation 결과가 필요합니다.


In [5]:
results = [
    {'filters': 4, 'kernel': 3, 'params': 554, 'one_step_loss': 0.71},
    {'filters': 8, 'kernel': 3, 'params': 1106, 'one_step_loss': 0.63},
    {'filters': 8, 'kernel': 5, 'params': 1234, 'one_step_loss': 0.67},
]
# TODO: one_step_loss가 가장 작은 현재 기록을 찾으세요.
lowest_step_record = min(results, key=lambda x: x['one_step_loss'])
print(lowest_step_record)


{'filters': 8, 'kernel': 3, 'params': 1106, 'one_step_loss': 0.63}


### 해설 및 실행 결과 해석

- lowest_step_record는 현재 적힌 one-step loss 중 최솟값을 찾은 결과일 뿐 best 모델이 아닙니다. 모델을 선택하려면 같은 조건으로 충분히 학습하고 validation loss 같은 선택 metric을 비교해야 합니다.
